In [ ]:
from __future__ import annotations

import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import softmax
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu, spearmanr

from sklearn.decomposition import PCA
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

import hashlib
import json
import shutil

import joblib
import statsmodels.api as sm

from IPython.display import display

In [ ]:
DATA_DIR = Path("data")

SOURCE_OUTPUT_DIR = DATA_DIR / "celligner_mss_tmbh_screen"
OUTPUT_DIR = DATA_DIR / "celligner_mss_tmbh_pan_cancer_screen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEPMAP_EXPRESSION_PATH = DATA_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
DEPMAP_MODEL_PATH = DATA_DIR / "Model.csv"
PRISM_RESPONSE_PATH = DATA_DIR / "secondary-screen-dose-response-curve-parameters.csv"
PRISM_TREATMENT_INFO_PATH = DATA_DIR / "secondary-screen-replicate-collapsed-treatment-info.csv"

TUMOR_DATASETS = {
    "CRC": {
        "expression_path": DATA_DIR / "CRC_TCGA_LOG2TPM.csv",
        "metadata_path": DATA_DIR / "clinical_data.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
    "UCEC": {
        "expression_path": DATA_DIR / "UCEC_TCGA_LOG2TPM.csv",
        "metadata_path": DATA_DIR / "ucec_clinical.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
}

EXTRA_ALIGNMENT_DATASETS = []

CELLIGNER_MODE = "load"
CELLIGNER_ALIGNED_PATH = SOURCE_OUTPUT_DIR / "celligner_aligned_expression.pkl"
CELLIGNER_MODEL_PATH = SOURCE_OUTPUT_DIR / "celligner_model.pkl"
CELLIGNER_LOW_MEMORY = True
RUN_ALIGNMENT_PCA = False

CLASS_ORDER = ["MSS/TMB-L", "MSS/TMB-H", "MSI-H"]

K_CANDIDATES = [1, 3, 5, 7]
TEMPERATURE_CANDIDATES = [0.03, 0.05, 0.10, 0.20, 0.50]
PCA_VARIANCE = 0.90
MAX_VARIABLE_FEATURES = 5000
CV_REPEATS = 10

PRIMARY_PROBABILITY_THRESHOLD = 0.60
PRIMARY_MARGIN_THRESHOLD = 0.10
THRESHOLDS_TO_TEST = [0.50, 0.60, 0.70]

MIN_MODELS_PER_TISSUE = 4
MIN_TOTAL_MODELS_CONTINUOUS = 30
MIN_TISSUES_CONTINUOUS = 3
MIN_MODELS_PER_HARD_GROUP = 3
MIN_TISSUES_HARD = 2
FDR_CUTOFF = 0.05

CRC_ONCOTREE_CODES = ["COAD", "READ", "MACR", "COADREAD"]
UTERINE_CARCINOMA_CODES = ["UCEC", "UCS", "UCCC", "UASC"]
UTERINE_BROAD_CODES = ["UCEC", "UCS", "UCCC", "UASC", "USARC", "ESS", "ULMS"]

SOURCE_CODES = {
    "CRC": set(CRC_ONCOTREE_CODES),
    "UCEC": set(UTERINE_BROAD_CODES),
}

CRC_FIXED_PARAMETERS = (1, 0.05)
CRC_KNOWN_METRICS = {
    "balanced_accuracy": 0.614235,
    "macro_f1": 0.623277,
    "macro_ovr_auc": 0.913421,
}

SOURCE_THRESHOLDS = {
    "CRC": {
        "confidence": 0.60,
        "margin": 0.10,
    },
    "UCEC": {
        "confidence": None,
        "margin": 0.05,
    },
}

print("Source cache:", SOURCE_OUTPUT_DIR)
print("Pan-cancer output:", OUTPUT_DIR)

In [ ]:
def read_table(path: Path, index_col: Optional[int] = 0) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    suffix = "".join(path.suffixes).lower()
    if suffix.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffix.endswith((".pkl", ".pickle")):
        return pd.read_pickle(path)
    if suffix.endswith((".tsv", ".txt")):
        return pd.read_csv(path, sep="\t", index_col=index_col)
    return pd.read_csv(path, index_col=index_col)


def canonical_sample_id(value: object) -> str:
    value = str(value).strip().replace(".", "-")
    if value.upper().startswith("TCGA-") and len(value) >= 16:
        value = value[:16]
    return value


def clean_gene_id(value: object) -> str:
    value = str(value).strip()
    if " (" in value:
        value = value.split(" (", 1)[0]
    if "|" in value:
        value = value.split("|", 1)[0]
    if value.upper().startswith("ENSG") and "." in value:
        value = value.split(".", 1)[0]
    return value.upper()


def collapse_duplicate_genes(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x.columns = [clean_gene_id(c) for c in x.columns]
    return x.T.groupby(level=0).mean().T


def normalize_group_label(value: object):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().upper().replace("_", "-").replace(" ", "")
    mapping = {
        "MSS/TMB-L": "MSS/TMB-L", "MSS-TMB-L": "MSS/TMB-L", "MSS/TMBL": "MSS/TMB-L",
        "MSS/TMB-H": "MSS/TMB-H", "MSS-TMB-H": "MSS/TMB-H", "MSS/TMBH": "MSS/TMB-H",
        "MSI-H": "MSI-H", "MSIH": "MSI-H", "MSI/H": "MSI-H",
    }
    return mapping.get(key, str(value).strip())


def load_metadata(path, cancer, group_col, sample_id_col=None):
    meta = read_table(path, index_col=0 if sample_id_col is None else None)
    ids = pd.Series(meta.index, index=meta.index) if sample_id_col is None else meta[sample_id_col]
    if group_col not in meta.columns:
        raise KeyError(f"{group_col!r} not found in {path}. Columns: {list(meta.columns)}")
    out = pd.DataFrame({
        "sample_id": ids.map(canonical_sample_id).values,
        "cancer": cancer,
        "group": meta[group_col].map(normalize_group_label).values,
    })
    out = out[out["group"].isin(CLASS_ORDER)].drop_duplicates("sample_id")
    out["aligned_id"] = "TCGA::" + out["sample_id"]
    return out.set_index("sample_id", drop=False)


def orient_to_samples(x: pd.DataFrame, expected_ids=None):
    x = x.copy()
    x.index = x.index.map(canonical_sample_id)
    x.columns = [canonical_sample_id(c) for c in x.columns]
    if expected_ids is None:
        return x
    expected = {canonical_sample_id(i) for i in expected_ids}
    row_overlap = len(expected.intersection(x.index))
    col_overlap = len(expected.intersection(x.columns))
    if col_overlap > row_overlap:
        x = x.T
        x.index = x.index.map(canonical_sample_id)
    if max(row_overlap, col_overlap) == 0:
        warnings.warn("No metadata IDs matched either expression axis; assuming samples are rows.")
    return x


def load_tumor_expression(path, metadata_ids, already_log2_tpm):
    x = orient_to_samples(read_table(path, index_col=0), metadata_ids)
    x = x.apply(pd.to_numeric, errors="coerce")
    if not already_log2_tpm:
        x = np.log2(x.clip(lower=0) + 1)
    x = collapse_duplicate_genes(x)
    return x.groupby(level=0).mean()


def load_depmap_expression(path):
    x = read_table(path, index_col=0)
    if "MODELID" in x.columns:
        x = x.set_index("MODELID")
    elif "ModelID" in x.columns:
        x = x.set_index("ModelID")
    x.index = x.index.astype(str)
    x = x.drop(columns=["MODELID", "ModelID"], errors="ignore")
    x = x.apply(pd.to_numeric, errors="coerce")
    return collapse_duplicate_genes(x).groupby(level=0).mean()

In [ ]:
depmap_expression = load_depmap_expression(DEPMAP_EXPRESSION_PATH)
models = read_table(DEPMAP_MODEL_PATH, index_col=None)
prism = read_table(PRISM_RESPONSE_PATH, index_col=None)
treatment_info = read_table(PRISM_TREATMENT_INFO_PATH, index_col=None)

required_model_columns = {"ModelID", "OncotreeCode"}
missing = required_model_columns.difference(models.columns)
if missing:
    raise KeyError(f"Missing Model.csv columns: {sorted(missing)}")

tumor_expression_parts = []
tumor_metadata_parts = []

for cancer, config in TUMOR_DATASETS.items():
    meta = load_metadata(
        config["metadata_path"], cancer, config["group_col"], config["sample_id_col"]
    )
    expr = load_tumor_expression(
        config["expression_path"], meta["sample_id"], config["already_log2_tpm"]
    )
    common_ids = meta.index.intersection(expr.index)
    if len(common_ids) == 0:
        raise ValueError(f"No matched metadata/expression samples for {cancer}.")
    meta = meta.loc[common_ids].copy()
    expr = expr.loc[common_ids].copy()
    expr.index = "TCGA::" + expr.index
    tumor_metadata_parts.append(meta)
    tumor_expression_parts.append(expr)

tumor_metadata = pd.concat(tumor_metadata_parts)
labeled_tumor_expression = pd.concat(tumor_expression_parts, axis=0, join="outer")

extra_parts = []
for config in EXTRA_ALIGNMENT_DATASETS:
    extra = load_tumor_expression(
        config["expression_path"], None, config["already_log2_tpm"]
    )
    extra.index = "EXTRA::" + str(config["name"]) + "::" + extra.index.astype(str)
    extra_parts.append(extra)

all_tumor_expression = pd.concat(
    [labeled_tumor_expression] + extra_parts, axis=0, join="outer"
)

print("DepMap:", depmap_expression.shape)
print("Labeled tumors:", labeled_tumor_expression.shape)
print("All alignment tumors:", all_tumor_expression.shape)
display(tumor_metadata.groupby(["cancer", "group"]).size().rename("n").reset_index())

In [ ]:
common_genes = depmap_expression.columns.intersection(all_tumor_expression.columns)
if len(common_genes) < 1000:
    raise ValueError(
        f"Only {len(common_genes)} overlapping genes. Harmonize gene identifiers first."
    )

ccle_input = depmap_expression.loc[:, common_genes].copy()
ccle_input.index = "CL::" + ccle_input.index.astype(str)
tumor_input = all_tumor_expression.loc[:, common_genes].copy()

ccle_input = ccle_input.fillna(ccle_input.median())
tumor_input = tumor_input.fillna(tumor_input.median())
usable = (
    ccle_input.notna().all() & tumor_input.notna().all()
    & ((ccle_input.var() > 0) | (tumor_input.var() > 0))
)
ccle_input = ccle_input.loc[:, usable]
tumor_input = tumor_input.loc[:, usable]

expected_ids = set(ccle_input.index).union(tumor_input.index)

def normalize_aligned_output(x):
    x = pd.DataFrame(x).copy()
    row_overlap = len(expected_ids.intersection(x.index.astype(str)))
    col_overlap = len(expected_ids.intersection(x.columns.astype(str)))
    if col_overlap > row_overlap:
        x = x.T
    x.index = x.index.astype(str)
    return x.apply(pd.to_numeric, errors="coerce")

if CELLIGNER_MODE == "load" and not CELLIGNER_ALIGNED_PATH.exists():
    raise FileNotFoundError(
        f"Saved Celligner alignment not found: {CELLIGNER_ALIGNED_PATH}. "
        "Run the original notebook first or set CELLIGNER_MODE='run'."
    )

if CELLIGNER_MODE == "run":
    try:
        from celligner import Celligner
    except ImportError as exc:
        raise ImportError(
            "Celligner is not installed. Install it or set CELLIGNER_MODE='load'."
        ) from exc
    try:
        aligner = Celligner(low_memory=CELLIGNER_LOW_MEMORY)
    except TypeError:
        aligner = Celligner()
        if hasattr(aligner, "low_memory"):
            aligner.low_memory = CELLIGNER_LOW_MEMORY
    aligner.fit(ccle_input)
    aligner.transform(tumor_input)
    aligned_expression = normalize_aligned_output(aligner.combined_output)
    aligned_expression.to_pickle(CELLIGNER_ALIGNED_PATH)
    try:
        aligner.save(str(CELLIGNER_MODEL_PATH))
    except Exception as exc:
        warnings.warn(f"Could not save Celligner model: {exc}")
elif CELLIGNER_MODE == "load":
    aligned_expression = normalize_aligned_output(pd.read_pickle(CELLIGNER_ALIGNED_PATH))
else:
    raise ValueError("CELLIGNER_MODE must be 'run' or 'load'.")

print("Aligned matrix:", aligned_expression.shape)
print("Expected samples missing:", len(expected_ids.difference(aligned_expression.index)))

In [ ]:
def plot_source_pca(matrix, max_features=3000):
    source = pd.Series(index=matrix.index, dtype=object)
    source.loc[source.index.str.startswith("CL::")] = "Cell line"
    source.loc[source.index.str.startswith("TCGA::")] = "TCGA tumor"
    source.loc[source.index.str.startswith("EXTRA::")] = "Other tumor"

    ids = source.dropna().index.intersection(matrix.index)
    x = matrix.loc[ids].copy()
    features = x.var().sort_values(ascending=False).head(max_features).index
    x = x[features].fillna(x[features].median())
    coordinates = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(x)
    )

    plt.figure(figsize=(8, 6))
    for label in source.loc[ids].unique():
        mask = source.loc[ids].values == label
        plt.scatter(coordinates[mask, 0], coordinates[mask, 1], s=18, alpha=0.65, label=label)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Celligner-aligned expression")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

if RUN_ALIGNMENT_PCA:
    plot_source_pca(aligned_expression)

In [ ]:
@dataclass
class BalancedNearestGroupClassifier:
    k: int = 3
    temperature: float = 0.10
    pca_variance: float = 0.90
    max_features: int = 5000
    class_order: tuple = tuple(CLASS_ORDER)

    def fit(self, x, y):
        x = x.copy().apply(pd.to_numeric, errors="coerce")
        y = pd.Series(y, index=x.index).astype(str)
        missing = set(self.class_order).difference(set(y))
        if missing:
            raise ValueError(f"Training data lack classes: {sorted(missing)}")

        variances = x.var().sort_values(ascending=False)
        self.features_ = variances[np.isfinite(variances) & (variances > 0)].head(
            self.max_features
        ).index.tolist()
        self.medians_ = x[self.features_].median()
        imputed = x[self.features_].fillna(self.medians_)
        self.scaler_ = StandardScaler()
        scaled = self.scaler_.fit_transform(imputed)
        self.pca_ = PCA(n_components=self.pca_variance, svd_solver="full")
        self.training_coordinates_ = self.pca_.fit_transform(scaled)
        self.training_labels_ = y.to_numpy()
        return self

    def _transform(self, x):
        x = x.copy().apply(pd.to_numeric, errors="coerce")
        for feature in set(self.features_).difference(x.columns):
            x[feature] = np.nan
        scaled = self.scaler_.transform(x[self.features_].fillna(self.medians_))
        return self.pca_.transform(scaled)

    def class_distances(self, x):
        distances = cdist(self._transform(x), self.training_coordinates_, metric="cosine")
        outputs = []
        for class_name in self.class_order:
            class_values = distances[:, self.training_labels_ == class_name]
            k_used = min(self.k, class_values.shape[1])
            nearest = np.partition(class_values, kth=k_used - 1, axis=1)[:, :k_used]
            outputs.append(nearest.mean(axis=1))
        return pd.DataFrame(np.column_stack(outputs), index=x.index, columns=self.class_order)

    def predict_proba(self, x):
        distances = self.class_distances(x)
        probabilities = softmax(-distances.to_numpy() / self.temperature, axis=1)
        return pd.DataFrame(probabilities, index=x.index, columns=self.class_order)


def make_cv(y):
    n_splits = min(5, int(y.value_counts().min()))
    if n_splits < 2:
        raise ValueError("At least two tumors per class are required.")
    return RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=CV_REPEATS, random_state=RANDOM_STATE
    )


def tune_classifier(x, y):
    rows = []
    cv = make_cv(y)
    for k in K_CANDIDATES:
        for temperature in TEMPERATURE_CANDIDATES:
            ba, mf1, losses = [], [], []
            for train, test in cv.split(x, y):
                model = BalancedNearestGroupClassifier(
                    k=k, temperature=temperature,
                    pca_variance=PCA_VARIANCE,
                    max_features=MAX_VARIABLE_FEATURES,
                ).fit(x.iloc[train], y.iloc[train])
                probs = model.predict_proba(x.iloc[test])
                pred = probs.idxmax(axis=1)
                ba.append(balanced_accuracy_score(y.iloc[test], pred))
                mf1.append(f1_score(y.iloc[test], pred, labels=CLASS_ORDER,
                                    average="macro", zero_division=0))
                metric_labels = sorted(CLASS_ORDER)
                losses.append(
                    log_loss(
                        y.iloc[test],
                        probs[metric_labels],
                        labels=metric_labels,
                    )
                )
            rows.append({
                "k": k, "temperature": temperature,
                "balanced_accuracy": np.mean(ba),
                "macro_f1": np.mean(mf1),
                "log_loss": np.mean(losses),
            })
    return pd.DataFrame(rows).sort_values(
        ["balanced_accuracy", "macro_f1", "log_loss"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


def repeated_oof(x, y, k, temperature):
    cv = make_cv(y)
    total = pd.DataFrame(0.0, index=x.index, columns=CLASS_ORDER)
    counts = pd.Series(0, index=x.index, dtype=int)
    for train, test in cv.split(x, y):
        model = BalancedNearestGroupClassifier(
            k=k, temperature=temperature,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        ).fit(x.iloc[train], y.iloc[train])
        probs = model.predict_proba(x.iloc[test])
        total.loc[probs.index] += probs
        counts.loc[probs.index] += 1
    return total.div(counts, axis=0)

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.special import softmax
from scipy.spatial.distance import cdist

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold


CACHE_DIR = SOURCE_OUTPUT_DIR / "classifier_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def atomic_joblib_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(obj, temporary, compress=3)
    temporary.replace(path)


def atomic_to_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.name + ".tmp")
    df.to_csv(temporary, index=index)
    temporary.replace(path)


def data_signature(x, y):
    """
    Create a cache identifier for the training dataset and settings.

    This prevents accidentally loading a cache generated from different
    samples, genes, or CV settings.
    """
    digest = hashlib.sha1()

    digest.update(str(x.shape).encode())
    digest.update("\n".join(x.index.astype(str)).encode())
    digest.update("\n".join(x.columns.astype(str)).encode())
    digest.update("\n".join(y.astype(str)).encode())

    configuration = {
        "class_order": list(CLASS_ORDER),
        "k_candidates": list(K_CANDIDATES),
        "cv_repeats": CV_REPEATS,
        "pca_variance": PCA_VARIANCE,
        "max_variable_features": MAX_VARIABLE_FEATURES,
        "random_state": RANDOM_STATE,
    }

    digest.update(
        json.dumps(configuration, sort_keys=True).encode()
    )

    return digest.hexdigest()[:16]

def tumor_training_data(cancer):
    meta = tumor_metadata[
        tumor_metadata["cancer"] == cancer
    ].copy()

    ids = meta.loc[
        meta["aligned_id"].isin(aligned_expression.index),
        "aligned_id",
    ]

    x = aligned_expression.loc[ids].copy()

    y = (
        meta.set_index("aligned_id")
        .loc[ids, "group"]
        .astype(str)
    )
    y.index = x.index

    print(
        f"{cancer}: {x.shape[0]} tumors × "
        f"{x.shape[1]} expression features"
    )
    print(y.value_counts())

    return x, y


def selected_cell_lines(codes):
    selected = (
        models[models["OncotreeCode"].isin(codes)]
        .drop_duplicates("ModelID", keep="first")
        .copy()
    )

    selected["aligned_id"] = (
        "CL::" + selected["ModelID"].astype(str)
    )

    selected = selected[
        selected["aligned_id"].isin(aligned_expression.index)
    ].copy()

    x = aligned_expression.loc[selected["aligned_id"]].copy()
    x.index = selected["aligned_id"].to_numpy()

    return x, selected

def make_cv(y):
    minimum_class_size = int(y.value_counts().min())
    n_splits = min(5, minimum_class_size)

    if n_splits < 2:
        raise ValueError(
            "At least two tumors per class are required."
        )

    return RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=CV_REPEATS,
        random_state=RANDOM_STATE,
    )


def distances_for_all_k(
    fitted_model,
    x_test,
    k_values,
):
    """
    Transform the test samples once, calculate pairwise distances once,
    and derive class distances for every k.
    """
    test_coordinates = fitted_model._transform(x_test)

    pairwise_distances = cdist(
        test_coordinates,
        fitted_model.training_coordinates_,
        metric="cosine",
    )

    pairwise_distances = np.nan_to_num(
        pairwise_distances,
        nan=1.0,
        posinf=1.0,
        neginf=0.0,
    )

    results = {}

    for k in k_values:
        class_distances = []

        for class_name in CLASS_ORDER:
            class_mask = (
                fitted_model.training_labels_ == class_name
            )

            distances = pairwise_distances[:, class_mask]

            if distances.shape[1] == 0:
                raise ValueError(
                    f"No training samples for {class_name}"
                )

            k_used = min(int(k), distances.shape[1])

            nearest = np.partition(
                distances,
                kth=k_used - 1,
                axis=1,
            )[:, :k_used]

            class_distances.append(nearest.mean(axis=1))

        results[int(k)] = np.column_stack(class_distances)

    return results


def build_cv_distance_cache(
    cancer,
    x,
    y,
    force=False,
):
    """
    Build or recover one cache file per cross-validation fold.

    After an interruption, completed folds are loaded and only unfinished
    folds are calculated.
    """
    signature = data_signature(x, y)

    fold_dir = (
        CACHE_DIR
        / cancer.lower()
        / signature
        / "cv_folds"
    )
    fold_dir.mkdir(parents=True, exist_ok=True)

    cv = make_cv(y)
    splits = list(cv.split(x, y))

    fold_results = []

    print(
        f"{cancer}: {len(splits)} CV folds; "
        f"cache directory: {fold_dir}"
    )

    for fold_number, (train_idx, test_idx) in enumerate(splits):
        fold_path = fold_dir / f"fold_{fold_number:03d}.joblib"

        expected_test_ids = x.index[test_idx].astype(str).tolist()

        if fold_path.exists() and not force:
            try:
                payload = joblib.load(fold_path)

                valid = (
                    payload.get("signature") == signature
                    and payload.get("test_ids")
                    == expected_test_ids
                )

                if valid:
                    fold_results.append(payload)
                    print(
                        f"Fold {fold_number + 1}/{len(splits)}: "
                        "loaded"
                    )
                    continue

                print(
                    f"Fold {fold_number + 1}: "
                    "cache did not match; recalculating"
                )

            except Exception as error:
                print(
                    f"Fold {fold_number + 1}: "
                    f"could not load cache ({error}); recalculating"
                )

        print(
            f"Fold {fold_number + 1}/{len(splits)}: fitting"
        )

        model = BalancedNearestGroupClassifier(
            k=1,
            temperature=1.0,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        )

        model.fit(
            x.iloc[train_idx],
            y.iloc[train_idx],
        )

        distances_by_k = distances_for_all_k(
            fitted_model=model,
            x_test=x.iloc[test_idx],
            k_values=K_CANDIDATES,
        )

        payload = {
            "signature": signature,
            "fold_number": fold_number,
            "test_ids": expected_test_ids,
            "y_true": y.iloc[test_idx].astype(str).tolist(),
            "distances_by_k": distances_by_k,
        }

        
        atomic_joblib_dump(payload, fold_path)
        fold_results.append(payload)

    return fold_results, signature

def tune_from_cached_distances(
    cancer,
    folds,
    signature,
    force=False,
):
    result_path = (
        CACHE_DIR
        / cancer.lower()
        / signature
        / "tuning.csv"
    )

    if result_path.exists() and not force:
        print(f"Loading cached tuning: {result_path}")
        return pd.read_csv(result_path)

    rows = []

    for k in K_CANDIDATES:
        for temperature in TEMPERATURE_CANDIDATES:
            balanced_accuracy_values = []
            macro_f1_values = []
            log_loss_values = []

            for fold in folds:
                distances = fold["distances_by_k"][int(k)]
                y_true = np.asarray(fold["y_true"])

                probabilities = softmax(
                    -distances / float(temperature),
                    axis=1,
                )

                predictions = np.asarray(CLASS_ORDER)[
                    probabilities.argmax(axis=1)
                ]

                balanced_accuracy_values.append(
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    )
                )

                macro_f1_values.append(
                    f1_score(
                        y_true,
                        predictions,
                        labels=CLASS_ORDER,
                        average="macro",
                        zero_division=0,
                    )
                )

                log_loss_values.append(
                    log_loss(
                        y_true,
                        probabilities,
                        labels=CLASS_ORDER,
                    )
                )

            rows.append({
                "k": int(k),
                "temperature": float(temperature),
                "balanced_accuracy": np.mean(
                    balanced_accuracy_values
                ),
                "macro_f1": np.mean(macro_f1_values),
                "log_loss": np.mean(log_loss_values),
            })

    tuning = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "balanced_accuracy",
                "macro_f1",
                "log_loss",
            ],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    atomic_to_csv(tuning, result_path, index=False)

    return tuning

def oof_from_cached_distances(
    x,
    folds,
    k,
    temperature,
):
    probability_sum = pd.DataFrame(
        0.0,
        index=x.index,
        columns=CLASS_ORDER,
    )

    prediction_count = pd.Series(
        0,
        index=x.index,
        dtype=int,
    )

    for fold in folds:
        test_ids = fold["test_ids"]
        distances = fold["distances_by_k"][int(k)]

        probabilities = softmax(
            -distances / float(temperature),
            axis=1,
        )

        probability_sum.loc[
            test_ids,
            CLASS_ORDER,
        ] += probabilities

        prediction_count.loc[test_ids] += 1

    if (prediction_count == 0).any():
        missing = prediction_count[
            prediction_count == 0
        ].index.tolist()

        raise RuntimeError(
            f"Samples without OOF predictions: {missing}"
        )

    return probability_sum.div(
        prediction_count,
        axis=0,
    )

def train_and_predict_cached(
    cancer,
    codes,
    fixed_parameters=None,
    calculate_oof=True,
    known_metrics=None,
    force=False,
):
    """
    Parameters
    ----------
    fixed_parameters
        Optional tuple: (k, temperature). When supplied, hyperparameter
        tuning is skipped.

    calculate_oof
        If False, skip repeated CV and only fit the final model.

    known_metrics
        Optional dictionary containing previously calculated metrics,
        useful when calculate_oof=False.
    """
    x_tumor, y_tumor = tumor_training_data(cancer)
    signature = data_signature(x_tumor, y_tumor)

    analysis_dir = (
        CACHE_DIR
        / cancer.lower()
        / signature
    )
    analysis_dir.mkdir(parents=True, exist_ok=True)

    folds = None
    tuning = None
    oof = None

    
    if fixed_parameters is None or calculate_oof:
        folds, signature = build_cv_distance_cache(
            cancer=cancer,
            x=x_tumor,
            y=y_tumor,
            force=force,
        )

    if fixed_parameters is None:
        tuning = tune_from_cached_distances(
            cancer=cancer,
            folds=folds,
            signature=signature,
            force=force,
        )

        best = tuning.iloc[0]
        k = int(best["k"])
        temperature = float(best["temperature"])

    else:
        k = int(fixed_parameters[0])
        temperature = float(fixed_parameters[1])

        tuning = pd.DataFrame([{
            "k": k,
            "temperature": temperature,
            "fixed_parameters": True,
        }])

    print(
        f"{cancer}: selected k={k}, "
        f"temperature={temperature}"
    )

    performance_path = analysis_dir / "performance.csv"
    oof_path = analysis_dir / (
        f"oof_k{k}_temperature{temperature}.pkl"
    )

    if calculate_oof:
        if oof_path.exists() and not force:
            print(f"Loading cached OOF: {oof_path}")
            oof = pd.read_pickle(oof_path)
        else:
            oof = oof_from_cached_distances(
                x=x_tumor,
                folds=folds,
                k=k,
                temperature=temperature,
            )

            temporary = oof_path.with_name(
                oof_path.name + ".tmp"
            )
            oof.to_pickle(temporary)
            temporary.replace(oof_path)

        oof_pred = oof.idxmax(axis=1)

        performance_values = {
            "cancer": cancer,
            "k": k,
            "temperature": temperature,
            "balanced_accuracy": balanced_accuracy_score(
                y_tumor,
                oof_pred,
            ),
            "macro_f1": f1_score(
                y_tumor,
                oof_pred,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0,
            ),
        }

        auc_order = sorted(CLASS_ORDER)

        try:
            performance_values["macro_ovr_auc"] = (
                roc_auc_score(
                    y_tumor,
                    oof[auc_order],
                    labels=auc_order,
                    multi_class="ovr",
                    average="macro",
                )
            )
        except ValueError:
            performance_values["macro_ovr_auc"] = np.nan

        performance = pd.DataFrame(
            [performance_values]
        )

        print(f"\n{cancer} repeated cross-validation")
        display(performance)

        display(
            pd.DataFrame(
                confusion_matrix(
                    y_tumor,
                    oof_pred,
                    labels=CLASS_ORDER,
                ),
                index=[
                    f"true_{value}"
                    for value in CLASS_ORDER
                ],
                columns=[
                    f"pred_{value}"
                    for value in CLASS_ORDER
                ],
            )
        )

        print(
            classification_report(
                y_tumor,
                oof_pred,
                labels=CLASS_ORDER,
                zero_division=0,
            )
        )

        oof_export = oof.copy()
        oof_export["true_group"] = y_tumor
        oof_export["predicted_group"] = oof_pred
        oof_export["cancer"] = cancer
        oof_export.index.name = "aligned_id"
        oof_export = oof_export.reset_index()

        atomic_to_csv(
            oof_export,
            analysis_dir / "oof_predictions.csv",
            index=False,
        )

    else:
        performance_values = {
            "cancer": cancer,
            "k": k,
            "temperature": temperature,
            "balanced_accuracy": np.nan,
            "macro_f1": np.nan,
            "macro_ovr_auc": np.nan,
        }

        if known_metrics is not None:
            performance_values.update(known_metrics)

        performance = pd.DataFrame(
            [performance_values]
        )
        oof_export = pd.DataFrame()

    atomic_to_csv(
        performance,
        performance_path,
        index=False,
    )

    
    
    

    model_path = analysis_dir / (
        f"final_model_k{k}_temperature{temperature}.joblib"
    )

    if model_path.exists() and not force:
        try:
            print(f"Loading final model: {model_path}")
            final_model = joblib.load(model_path)
        except Exception as error:
            print(
                f"Could not load final model ({error}); refitting"
            )
            final_model = None
    else:
        final_model = None

    if final_model is None:
        print(f"{cancer}: fitting final model")

        final_model = BalancedNearestGroupClassifier(
            k=k,
            temperature=temperature,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        ).fit(
            x_tumor,
            y_tumor,
        )

        atomic_joblib_dump(
            final_model,
            model_path,
        )

    
    
    

    code_hash = hashlib.sha1(
        ",".join(sorted(codes)).encode()
    ).hexdigest()[:8]

    predictions_path = analysis_dir / (
        f"cell_line_predictions_{code_hash}.csv"
    )

    if predictions_path.exists() and not force:
        print(
            f"Loading cell-line predictions: "
            f"{predictions_path}"
        )
        predictions = pd.read_csv(predictions_path)

    else:
        x_cell, selected = selected_cell_lines(codes)

        probabilities = final_model.predict_proba(x_cell)

        predictions = probabilities.rename(
            columns={
                "MSS/TMB-L": "p_MSS_TMBL",
                "MSS/TMB-H": "p_MSS_TMBH",
                "MSI-H": "p_MSIH",
            }
        )

        predictions["predicted_group"] = (
            probabilities.idxmax(axis=1)
        )

        sorted_probabilities = np.sort(
            probabilities.to_numpy(),
            axis=1,
        )

        predictions["confidence"] = (
            sorted_probabilities[:, -1]
        )

        predictions["probability_margin"] = (
            sorted_probabilities[:, -1]
            - sorted_probabilities[:, -2]
        )

        predictions["DepMap_ID"] = (
            predictions.index
            .str.replace(
                r"^CL::",
                "",
                regex=True,
            )
        )

        
        selected_metadata = (
            selected
            .drop(columns="aligned_id")
            .rename(columns={"ModelID": "DepMap_ID"})
        )

        predictions = predictions.merge(
            selected_metadata,
            on="DepMap_ID",
            how="left",
            validate="one_to_one",
        )

        predictions.insert(
            0,
            "classifier_source",
            cancer,
        )

        atomic_to_csv(
            predictions,
            predictions_path,
            index=False,
        )

    return {
        "model": final_model,
        "tuning": tuning,
        "performance": performance,
        "oof": oof_export,
        "predictions": predictions,
        "cache_directory": analysis_dir,
    }

In [ ]:
crc_analysis = train_and_predict_cached(
    cancer="CRC",
    codes=CRC_ONCOTREE_CODES,
    fixed_parameters=CRC_FIXED_PARAMETERS,
    calculate_oof=False,
    known_metrics=CRC_KNOWN_METRICS,
)

ucec_analysis = train_and_predict_cached(
    cancer="UCEC",
    codes=UTERINE_BROAD_CODES,
    fixed_parameters=None,
    calculate_oof=True,
)

classifier_performance = pd.concat(
    [crc_analysis["performance"], ucec_analysis["performance"]],
    ignore_index=True,
)
display(classifier_performance)

In [ ]:
ucec_oof = ucec_analysis["oof"].copy()

probability_columns = [
    "MSS/TMB-L",
    "MSS/TMB-H",
    "MSI-H",
]

sorted_probabilities = np.sort(
    ucec_oof[probability_columns].to_numpy(),
    axis=1,
)

ucec_oof["confidence"] = sorted_probabilities[:, -1]
ucec_oof["probability_margin"] = (
    sorted_probabilities[:, -1]
    - sorted_probabilities[:, -2]
)

ucec_oof["correct"] = (
    ucec_oof["predicted_group"]
    == ucec_oof["true_group"]
)

display(
    ucec_oof.groupby("correct")[
        ["confidence", "probability_margin"]
    ].describe()
)
UCEC_MARGIN_THRESHOLD = (
    ucec_oof.loc[
        ucec_oof["correct"],
        "probability_margin",
    ]
    .quantile(0.25)
)

print("UCEC margin threshold:", UCEC_MARGIN_THRESHOLD)

In [ ]:
from sklearn.metrics import precision_recall_curve

y_binary = (
    ucec_oof["true_group"] == "MSS/TMB-H"
).astype(int)

precision, recall, thresholds = precision_recall_curve(
    y_binary,
    ucec_oof["MSS/TMB-H"],
)

threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})

threshold_table["f1"] = (
    2
    * threshold_table["precision"]
    * threshold_table["recall"]
    / (
        threshold_table["precision"]
        + threshold_table["recall"]
    )
).fillna(0)

display(
    threshold_table.sort_values(
        "f1",
        ascending=False,
    ).head(20)
)

UCEC_MSS_TMBH_THRESHOLD = threshold_table.loc[
    threshold_table["f1"].idxmax(),
    "threshold",
]

In [ ]:
def predict_all_cell_lines(fitted_model, classifier_source):
    """Score every Model.csv line that is present in aligned expression."""
    selected = (
        models.dropna(subset=["ModelID"])
        .drop_duplicates("ModelID", keep="first")
        .copy()
    )
    selected["aligned_id"] = "CL::" + selected["ModelID"].astype(str)
    selected = selected[
        selected["aligned_id"].isin(aligned_expression.index)
    ].copy()

    if selected.empty:
        raise ValueError("No Model.csv cell lines matched aligned expression.")

    x_cell = aligned_expression.loc[selected["aligned_id"]].copy()
    x_cell.index = selected["aligned_id"].to_numpy()

    probabilities = fitted_model.predict_proba(x_cell)
    predictions = probabilities.rename(columns={
        "MSS/TMB-L": "p_MSS_TMBL",
        "MSS/TMB-H": "p_MSS_TMBH",
        "MSI-H": "p_MSIH",
    })

    predictions["predicted_group"] = probabilities.idxmax(axis=1)
    sorted_probabilities = np.sort(probabilities.to_numpy(), axis=1)
    predictions["confidence"] = sorted_probabilities[:, -1]
    predictions["probability_margin"] = (
        sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
    )
    predictions["mss_tmbh_margin"] = (
        predictions["p_MSS_TMBH"]
        - predictions[["p_MSS_TMBL", "p_MSIH"]].max(axis=1)
    )
    predictions["DepMap_ID"] = predictions.index.str.replace(
        r"^CL::", "", regex=True
    )

    metadata = (
        selected.drop(columns="aligned_id")
        .rename(columns={"ModelID": "DepMap_ID"})
    )
    predictions = predictions.merge(
        metadata,
        on="DepMap_ID",
        how="left",
        validate="one_to_one",
    )
    predictions.insert(0, "classifier_source", classifier_source)
    return predictions


crc_all_predictions = predict_all_cell_lines(crc_analysis["model"], "CRC")
ucec_all_predictions = predict_all_cell_lines(ucec_analysis["model"], "UCEC")

all_cell_predictions = pd.concat(
    [crc_all_predictions, ucec_all_predictions],
    ignore_index=True,
)

print("Unique aligned models:", all_cell_predictions["DepMap_ID"].nunique())
display(
    all_cell_predictions.groupby(
        ["classifier_source", "predicted_group"]
    ).size().rename("n").reset_index()
)

atomic_to_csv(
    crc_all_predictions,
    OUTPUT_DIR / "crc_all_cell_line_predictions.csv",
    index=False,
)
atomic_to_csv(
    ucec_all_predictions,
    OUTPUT_DIR / "ucec_all_cell_line_predictions.csv",
    index=False,
)
atomic_to_csv(
    all_cell_predictions,
    OUTPUT_DIR / "cell_line_subgroup_predictions.csv",
    index=False,
)

In [ ]:
def first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def prepare_prism(prism, treatment_info=None, response_col="auc"):
    required = {"depmap_id", "name", response_col}
    missing = required.difference(prism.columns)
    if missing:
        raise KeyError(f"Missing PRISM columns: {sorted(missing)}")

    data = prism.copy()
    data["drug_name"] = data["name"].astype(str).str.strip()
    data["drug_id"] = data["drug_name"].str.lower()
    data[response_col] = pd.to_numeric(data[response_col], errors="coerce")
    data = data.dropna(subset=["depmap_id", "drug_id", response_col])

    collapsed = data.groupby(["depmap_id", "drug_id"], as_index=False).agg(
        drug_name=("drug_name", "first"),
        auc=(response_col, "median"),
        n_measurements=(response_col, "size"),
    )

    columns = ["name", "broad_id", "moa", "target", "phase", "disease.area", "indication"]
    parts = []
    for source in [prism, treatment_info]:
        if source is None or "name" not in source.columns:
            continue
        available = [c for c in columns if c in source.columns]
        meta = source[available].copy()
        meta["drug_id"] = meta["name"].astype(str).str.strip().str.lower()
        parts.append(meta)

    if parts:
        meta = pd.concat(parts, ignore_index=True)
        meta = meta.groupby("drug_id", as_index=False).agg({
            c: first_non_null for c in meta.columns if c != "drug_id"
        }).rename(columns={"name": "drug_name_metadata"})
    else:
        meta = collapsed[["drug_id", "drug_name"]].drop_duplicates()

    return collapsed, meta


prism_collapsed, drug_metadata = prepare_prism(prism, treatment_info)
print("Unique compounds:", prism_collapsed["drug_id"].nunique())
print("Unique cell lines:", prism_collapsed["depmap_id"].nunique())

In [ ]:
def choose_tissue_column(frame):
    candidates = [
        "OncotreeLineage",
        "OncotreePrimaryDisease",
        "OncotreeSubtype",
        "OncotreeCode",
    ]
    for column in candidates:
        if column in frame.columns and frame[column].notna().sum() > 0:
            if frame[column].nunique(dropna=True) >= 2:
                return column
    raise KeyError(
        "Could not identify a usable tissue column. Available columns: "
        + ", ".join(frame.columns.astype(str))
    )


TISSUE_COLUMN = choose_tissue_column(all_cell_predictions)
print("Tissue adjustment column:", TISSUE_COLUMN)

prediction_tissue_coverage = (
    all_cell_predictions
    .dropna(subset=[TISSUE_COLUMN])
    .groupby(["classifier_source", TISSUE_COLUMN])
    .agg(
        n_models=("DepMap_ID", "nunique"),
        median_p_MSS_TMBH=("p_MSS_TMBH", "median"),
        max_p_MSS_TMBH=("p_MSS_TMBH", "max"),
    )
    .reset_index()
    .sort_values(["classifier_source", "n_models"], ascending=[True, False])
)
display(prediction_tissue_coverage.head(40))
atomic_to_csv(
    prediction_tissue_coverage,
    OUTPUT_DIR / "pan_cancer_tissue_coverage.csv",
    index=False,
)

prism_model_ids = set(prism_collapsed["depmap_id"].astype(str))
model_coverage = (
    all_cell_predictions[
        ["classifier_source", "DepMap_ID", TISSUE_COLUMN]
    ]
    .drop_duplicates()
    .assign(
        has_PRISM=lambda x: x["DepMap_ID"].astype(str).isin(prism_model_ids)
    )
)

model_coverage_summary = (
    model_coverage
    .groupby(["classifier_source", TISSUE_COLUMN, "has_PRISM"])
    ["DepMap_ID"]
    .nunique()
    .rename("n_models")
    .reset_index()
)
display(model_coverage_summary.head(40))
atomic_to_csv(
    model_coverage_summary,
    OUTPUT_DIR / "pan_cancer_model_coverage.csv",
    index=False,
)

In [ ]:
def bh_adjust(values):
    values = pd.Series(values, copy=True)
    adjusted = pd.Series(np.nan, index=values.index, dtype=float)
    mask = values.notna() & np.isfinite(values)
    if mask.any():
        adjusted.loc[mask] = multipletests(
            values.loc[mask].astype(float),
            method="fdr_bh",
        )[1]
    return adjusted


def within_tissue_zscore(values):
    values = pd.to_numeric(values, errors="coerce")
    sd = values.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=values.index)
    return (values - values.mean()) / sd


def add_fdr_columns(results, within_columns):
    if results.empty:
        return results
    results = results.copy()
    results["FDR_global"] = bh_adjust(results["pvalue"])
    results["FDR_within_analysis"] = (
        results.groupby(within_columns, dropna=False)["pvalue"]
        .transform(bh_adjust)
    )
    return results


def source_tissue_mask(frame, classifier_source):
    if "OncotreeCode" not in frame.columns:
        return pd.Series(False, index=frame.index)
    return frame["OncotreeCode"].isin(
        SOURCE_CODES.get(classifier_source, set())
    )


def prepare_drug_model_data(predictions, exclude_source_tissue=False):
    columns = [
        "DepMap_ID",
        "classifier_source",
        TISSUE_COLUMN,
        "OncotreeCode",
        "p_MSS_TMBH",
        "mss_tmbh_margin",
        "predicted_group",
        "confidence",
        "probability_margin",
    ]

    columns = list(dict.fromkeys(
        column for column in columns
        if column in predictions.columns
    ))

    scores = predictions[columns].drop_duplicates(
        ["classifier_source", "DepMap_ID"]
    )

    if exclude_source_tissue:
        parts = []

        for classifier_source, part in scores.groupby(
            "classifier_source",
            dropna=False,
            sort=False,
        ):
            parts.append(
                part.loc[
                    ~source_tissue_mask(
                        part,
                        classifier_source,
                    )
                ]
            )

        scores = pd.concat(parts, ignore_index=True)

    
    
    
    merged_parts = []

    for classifier_source, source_scores in scores.groupby(
        "classifier_source",
        dropna=False,
        sort=False,
    ):
        source_scores = source_scores.copy()

        duplicated_models = (
            source_scores["DepMap_ID"]
            .duplicated(keep=False)
        )

        if duplicated_models.any():
            duplicate_ids = (
                source_scores.loc[
                    duplicated_models,
                    "DepMap_ID",
                ]
                .astype(str)
                .unique()
                .tolist()
            )

            raise ValueError(
                f"Duplicate score rows for classifier "
                f"{classifier_source}: {duplicate_ids[:10]}"
            )

        source_merged = source_scores.merge(
            prism_collapsed,
            left_on="DepMap_ID",
            right_on="depmap_id",
            how="inner",
            validate="one_to_many",
        )

        merged_parts.append(source_merged)

    if not merged_parts:
        return pd.DataFrame()

    merged = pd.concat(
        merged_parts,
        ignore_index=True,
    )

    return merged.drop_duplicates(
        [
            "classifier_source",
            "drug_id",
            "DepMap_ID",
        ],
        keep="first",
    )
    columns = [
        "DepMap_ID",
        "classifier_source",
        TISSUE_COLUMN,
        "OncotreeCode",
        "p_MSS_TMBH",
        "mss_tmbh_margin",
        "predicted_group",
        "confidence",
        "probability_margin",
    ]
    columns = list(dict.fromkeys(
        column for column in columns if column in predictions.columns
    ))

    scores = predictions[columns].drop_duplicates(
        ["classifier_source", "DepMap_ID"]
    )

    if exclude_source_tissue:
        parts = []
        for classifier_source, part in scores.groupby("classifier_source"):
            parts.append(
                part.loc[~source_tissue_mask(part, classifier_source)]
            )
        scores = pd.concat(parts, ignore_index=True)

    merged = scores.merge(
        prism_collapsed,
        left_on="DepMap_ID",
        right_on="depmap_id",
        how="inner",
        validate="one_to_many",
    )
    return merged.drop_duplicates(
        ["classifier_source", "drug_id", "DepMap_ID"]
    )

In [ ]:
def screen_tissue_adjusted_continuous(
    predictions,
    analysis_set,
    exclude_source_tissue=False,
    score_column="p_MSS_TMBH",
):
    merged = prepare_drug_model_data(
        predictions,
        exclude_source_tissue=exclude_source_tissue,
    )
    rows = []

    for classifier_source, source_df in merged.groupby("classifier_source"):
        for drug_id, drug_df in source_df.groupby("drug_id"):
            data = drug_df[
                [
                    "DepMap_ID",
                    TISSUE_COLUMN,
                    score_column,
                    "auc",
                    "drug_name",
                ]
            ].dropna().copy()

            tissue_counts = (
                data.groupby(TISSUE_COLUMN)["DepMap_ID"]
                .transform("nunique")
            )
            data = data[tissue_counts >= MIN_MODELS_PER_TISSUE].copy()
            if data.empty:
                continue

            data["score_within_tissue"] = (
                data.groupby(TISSUE_COLUMN)[score_column]
                .transform(within_tissue_zscore)
            )
            data["auc_within_tissue"] = (
                data["auc"]
                - data.groupby(TISSUE_COLUMN)["auc"].transform("mean")
            )
            data = data.dropna(
                subset=["score_within_tissue", "auc_within_tissue"]
            )

            n_models = data["DepMap_ID"].nunique()
            n_tissues = data[TISSUE_COLUMN].nunique()

            if n_models < MIN_TOTAL_MODELS_CONTINUOUS:
                continue
            if n_tissues < MIN_TISSUES_CONTINUOUS:
                continue
            if data["score_within_tissue"].nunique() < 2:
                continue

            fit = sm.OLS(
                data["auc_within_tissue"].astype(float),
                data[["score_within_tissue"]].astype(float),
            ).fit(cov_type="HC3")

            rho, rho_pvalue = spearmanr(
                data["score_within_tissue"],
                data["auc_within_tissue"],
            )

            rows.append({
                "analysis_set": analysis_set,
                "classifier_source": classifier_source,
                "score": score_column,
                "drug_id": drug_id,
                "drug": data["drug_name"].iloc[0],
                "n_models": n_models,
                "n_tissues": n_tissues,
                "beta_tissue_adjusted": fit.params["score_within_tissue"],
                "standard_error_HC3": fit.bse["score_within_tissue"],
                "pvalue": fit.pvalues["score_within_tissue"],
                "rho_within_tissue": rho,
                "rho_pvalue": rho_pvalue,
                "median_auc": data["auc"].median(),
            })

    results = pd.DataFrame(rows)
    return add_fdr_columns(
        results,
        ["analysis_set", "classifier_source", "score"],
    )


continuous_parts = [
    screen_tissue_adjusted_continuous(
        all_cell_predictions,
        analysis_set="all_lineages",
        exclude_source_tissue=False,
        score_column="p_MSS_TMBH",
    ),
    screen_tissue_adjusted_continuous(
        all_cell_predictions,
        analysis_set="exclude_source_tissue",
        exclude_source_tissue=True,
        score_column="p_MSS_TMBH",
    ),
    screen_tissue_adjusted_continuous(
        all_cell_predictions,
        analysis_set="all_lineages",
        exclude_source_tissue=False,
        score_column="mss_tmbh_margin",
    ),
]

nonempty_continuous = [x for x in continuous_parts if not x.empty]
continuous_results = (
    pd.concat(nonempty_continuous, ignore_index=True)
    if nonempty_continuous else pd.DataFrame()
)

if not continuous_results.empty:
    continuous_results = continuous_results.merge(
        drug_metadata,
        on="drug_id",
        how="left",
    ).sort_values(["FDR_within_analysis", "pvalue"])

display(continuous_results.head(40))
atomic_to_csv(
    continuous_results,
    OUTPUT_DIR / "all_drug_continuous_probability_results.csv",
    index=False,
)
atomic_to_csv(
    continuous_results,
    OUTPUT_DIR / "pan_cancer_tissue_adjusted_continuous_results.csv",
    index=False,
)

In [ ]:
ucec_predictions = all_cell_predictions[
    all_cell_predictions["classifier_source"] == "UCEC"
].copy()

print(
    "UCEC model:",
    "k =", ucec_analysis["model"].k,
    "temperature =", ucec_analysis["model"].temperature,
)

display(
    ucec_predictions[
        [
            "p_MSS_TMBL",
            "p_MSS_TMBH",
            "p_MSIH",
            "confidence",
            "probability_margin",
            "predicted_group",
        ]
    ].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print(
    "Pass probability:",
    (
        ucec_predictions["confidence"]
        >= PRIMARY_PROBABILITY_THRESHOLD
    ).sum(),
)

print(
    "Pass margin:",
    (
        ucec_predictions["probability_margin"]
        >= PRIMARY_MARGIN_THRESHOLD
    ).sum(),
)

print(
    "Pass both:",
    (
        (
            ucec_predictions["confidence"]
            >= PRIMARY_PROBABILITY_THRESHOLD
        )
        & (
            ucec_predictions["probability_margin"]
            >= PRIMARY_MARGIN_THRESHOLD
        )
    ).sum(),
)
display(
    ucec_predictions[
        [
            "DepMap_ID",
            "OncotreeCode",
            "predicted_group",
            "p_MSS_TMBL",
            "p_MSS_TMBH",
            "p_MSIH",
            "confidence",
            "probability_margin",
        ]
    ]
    .sort_values("confidence", ascending=False)
    .head(30)
)

In [ ]:

HARD_SELECTION_RULES = {
    "CRC": {
        "mode": "three_class_confidence",
        "confidence_threshold": PRIMARY_PROBABILITY_THRESHOLD,
        "margin_threshold": PRIMARY_MARGIN_THRESHOLD,
    },
    "UCEC": {
        "mode": "binary_mss_tmbh_probability",
        "mss_tmbh_threshold": float(UCEC_MSS_TMBH_THRESHOLD),
        "comparator_margin_threshold": float(UCEC_MARGIN_THRESHOLD),
        
        
    },
}
def assign_source_specific_hard_groups(
    source_df,
    classifier_source,
    selection_rules,
):
    """
    Assign mutually exclusive hard groups using classifier-specific rules.

    CRC:
        Uses the original three-class confidence and margin filters.

    UCEC:
        MSS/TMB-H is selected using the OOF-calibrated p_MSS_TMBH
        threshold. Comparator groups are selected from the original
        argmax prediction, subject to a UCEC-specific margin threshold.
    """
    if classifier_source not in selection_rules:
        raise KeyError(
            f"No hard-group selection rule for {classifier_source}"
        )

    settings = selection_rules[classifier_source]
    mode = settings["mode"]

    result = source_df.copy()
    result["screening_group"] = pd.NA
    result["selection_rule"] = mode
    result["selection_threshold"] = np.nan
    result["selection_margin_threshold"] = np.nan

    if mode == "three_class_confidence":
        confidence_threshold = float(
            settings["confidence_threshold"]
        )
        margin_threshold = float(
            settings["margin_threshold"]
        )

        eligible = (
            result["confidence"] >= confidence_threshold
        ) & (
            result["probability_margin"] >= margin_threshold
        )

        result.loc[
            eligible,
            "screening_group",
        ] = result.loc[
            eligible,
            "predicted_group",
        ]

        result["selection_threshold"] = confidence_threshold
        result["selection_margin_threshold"] = margin_threshold

    elif mode == "binary_mss_tmbh_probability":
        mss_threshold = float(
            settings["mss_tmbh_threshold"]
        )
        comparator_margin = float(
            settings["comparator_margin_threshold"]
        )

        
        
        is_mss_tmbh = (
            result["p_MSS_TMBH"] >= mss_threshold
        )

        result.loc[
            is_mss_tmbh,
            "screening_group",
        ] = "MSS/TMB-H"

        
        
        
        
        is_comparator = (
            ~is_mss_tmbh
            & result["predicted_group"].isin(
                ["MSS/TMB-L", "MSI-H"]
            )
            & (
                result["probability_margin"]
                >= comparator_margin
            )
        )

        result.loc[
            is_comparator,
            "screening_group",
        ] = result.loc[
            is_comparator,
            "predicted_group",
        ]

        result["selection_threshold"] = mss_threshold
        result["selection_margin_threshold"] = comparator_margin

    else:
        raise ValueError(
            f"Unknown selection mode: {mode}"
        )

    return result.dropna(
        subset=["screening_group"]
    ).copy()

def screen_tissue_adjusted_hard(
    predictions,
    selection_rules,
    analysis_set="all_lineages",
):
    merged = prepare_drug_model_data(
        predictions,
        exclude_source_tissue=False,
    )

    selected_parts = []

    for classifier_source, source_df in merged.groupby(
        "classifier_source",
        sort=False,
    ):
        if classifier_source not in selection_rules:
            print(
                f"Skipping {classifier_source}: "
                "no selection rule supplied."
            )
            continue

        selected = assign_source_specific_hard_groups(
            source_df=source_df,
            classifier_source=classifier_source,
            selection_rules=selection_rules,
        )

        if selected.empty:
            print(
                f"{classifier_source}: no models passed "
                "hard-group selection."
            )
            continue

        selected_parts.append(selected)

    if not selected_parts:
        return pd.DataFrame()

    merged = pd.concat(
        selected_parts,
        ignore_index=True,
    )

    print("\nModels passing source-specific hard selection:")
    display(
        merged.groupby(
            ["classifier_source", "screening_group"]
        )["DepMap_ID"]
        .nunique()
        .unstack(fill_value=0)
    )

    rows = []

    for classifier_source, source_df in merged.groupby(
        "classifier_source",
        sort=False,
    ):
        source_settings = selection_rules[
            classifier_source
        ]

        for drug_id, drug_df in source_df.groupby(
            "drug_id",
            sort=False,
        ):
            for comparator in ["MSS/TMB-L", "MSI-H"]:
                data = drug_df[
                    drug_df["screening_group"].isin(
                        ["MSS/TMB-H", comparator]
                    )
                ][
                    [
                        "DepMap_ID",
                        TISSUE_COLUMN,
                        "screening_group",
                        "auc",
                        "drug_name",
                    ]
                ].dropna().copy()

                data = data.drop_duplicates(
                    ["DepMap_ID", "screening_group"]
                )

                if data.empty:
                    continue

                
                
                tissue_group_counts = (
                    data.groupby(TISSUE_COLUMN)[
                        "screening_group"
                    ]
                    .nunique()
                )

                eligible_tissues = tissue_group_counts[
                    tissue_group_counts == 2
                ].index

                data = data[
                    data[TISSUE_COLUMN].isin(
                        eligible_tissues
                    )
                ].copy()

                if data.empty:
                    continue

                n_mss = data.loc[
                    (
                        data["screening_group"]
                        == "MSS/TMB-H"
                    ),
                    "DepMap_ID",
                ].nunique()

                n_ref = data.loc[
                    (
                        data["screening_group"]
                        == comparator
                    ),
                    "DepMap_ID",
                ].nunique()

                n_tissues = data[
                    TISSUE_COLUMN
                ].nunique()

                if n_mss < MIN_MODELS_PER_HARD_GROUP:
                    continue

                if n_ref < MIN_MODELS_PER_HARD_GROUP:
                    continue

                if n_tissues < MIN_TISSUES_HARD:
                    continue

                data["mss_indicator"] = (
                    data["screening_group"]
                    == "MSS/TMB-H"
                ).astype(float)

                data["group_within_tissue"] = (
                    data["mss_indicator"]
                    - data.groupby(TISSUE_COLUMN)[
                        "mss_indicator"
                    ].transform("mean")
                )

                data["auc_within_tissue"] = (
                    data["auc"]
                    - data.groupby(TISSUE_COLUMN)[
                        "auc"
                    ].transform("mean")
                )

                if np.isclose(
                    data[
                        "group_within_tissue"
                    ].abs().sum(),
                    0,
                ):
                    continue

                fit = sm.OLS(
                    data["auc_within_tissue"].astype(float),
                    data[
                        ["group_within_tissue"]
                    ].astype(float),
                ).fit(cov_type="HC3")

                mss_auc = data.loc[
                    (
                        data["screening_group"]
                        == "MSS/TMB-H"
                    ),
                    "auc",
                ]

                ref_auc = data.loc[
                    (
                        data["screening_group"]
                        == comparator
                    ),
                    "auc",
                ]

                if (
                    source_settings["mode"]
                    == "three_class_confidence"
                ):
                    selection_threshold = (
                        source_settings[
                            "confidence_threshold"
                        ]
                    )
                    margin_threshold = (
                        source_settings[
                            "margin_threshold"
                        ]
                    )
                else:
                    selection_threshold = (
                        source_settings[
                            "mss_tmbh_threshold"
                        ]
                    )
                    margin_threshold = (
                        source_settings[
                            "comparator_margin_threshold"
                        ]
                    )

                rows.append({
                    "analysis_set": analysis_set,
                    "classifier_source": classifier_source,
                    "selection_rule": source_settings["mode"],
                    "selection_threshold": (
                        selection_threshold
                    ),
                    "selection_margin_threshold": (
                        margin_threshold
                    ),
                    "drug_id": drug_id,
                    "drug": data["drug_name"].iloc[0],
                    "contrast": (
                        f"MSS/TMB-H vs {comparator}"
                    ),
                    "n_MSS_TMBH": n_mss,
                    "n_comparator": n_ref,
                    "n_tissues": n_tissues,
                    "median_auc_MSS_TMBH": (
                        mss_auc.median()
                    ),
                    "median_auc_comparator": (
                        ref_auc.median()
                    ),
                    "median_auc_difference": (
                        mss_auc.median()
                        - ref_auc.median()
                    ),
                    "beta_tissue_adjusted": (
                        fit.params[
                            "group_within_tissue"
                        ]
                    ),
                    "standard_error_HC3": (
                        fit.bse[
                            "group_within_tissue"
                        ]
                    ),
                    "pvalue": fit.pvalues[
                        "group_within_tissue"
                    ],
                })

    results = pd.DataFrame(rows)

    if results.empty:
        return results

    return add_fdr_columns(
        results,
        [
            "analysis_set",
            "classifier_source",
            "selection_rule",
            "selection_threshold",
            "contrast",
        ],
    )

In [ ]:
threshold_parts = []

for threshold in THRESHOLDS_TO_TEST:
    result = screen_tissue_adjusted_hard(
        all_cell_predictions,
        threshold=threshold,
    )
    if not result.empty:
        threshold_parts.append(result)

threshold_results = (
    pd.concat(threshold_parts, ignore_index=True)
    if threshold_parts else pd.DataFrame()
)

if not threshold_results.empty:
    threshold_results["significant_sensitive"] = (
        (threshold_results["FDR_within_analysis"] < FDR_CUTOFF)
        & (threshold_results["beta_tissue_adjusted"] < 0)
    )

    threshold_summary = (
        threshold_results
        .groupby(
            ["classifier_source", "drug_id", "drug", "contrast"]
        )
        .agg(
            thresholds_tested=("probability_threshold", "nunique"),
            thresholds_significant=("significant_sensitive", "sum"),
            all_effects_sensitive=(
                "beta_tissue_adjusted",
                lambda x: bool((x < 0).all()),
            ),
            median_adjusted_effect=(
                "beta_tissue_adjusted",
                "median",
            ),
            maximum_FDR=("FDR_within_analysis", "max"),
        )
        .reset_index()
        .sort_values(
            ["thresholds_significant", "median_adjusted_effect"],
            ascending=[False, True],
        )
    )
else:
    threshold_summary = pd.DataFrame()

display(threshold_summary.head(50))

atomic_to_csv(
    threshold_results,
    OUTPUT_DIR / "all_drug_threshold_sensitivity_results.csv",
    index=False,
)
atomic_to_csv(
    threshold_summary,
    OUTPUT_DIR / "all_drug_threshold_sensitivity_summary.csv",
    index=False,
)

In [ ]:
primary_hard_results = screen_tissue_adjusted_hard(
    predictions=all_cell_predictions,
    selection_rules=HARD_SELECTION_RULES,
    analysis_set="all_lineages",
)

if not primary_hard_results.empty:
    primary_hard_results = (
        primary_hard_results
        .merge(
            drug_metadata,
            on="drug_id",
            how="left",
        )
        .sort_values(
            [
                "FDR_within_analysis",
                "pvalue",
            ]
        )
    )

display(
    primary_hard_results.groupby(
        ["classifier_source", "contrast"]
    ).size()
)

display(primary_hard_results.head(40))

atomic_to_csv(
    primary_hard_results,
    OUTPUT_DIR / "all_drug_hard_group_results.csv",
    index=False,
)

In [ ]:
def build_continuous_candidates(results):
    if results.empty:
        return pd.DataFrame()

    primary = results[
        (results["analysis_set"] == "all_lineages")
        & (results["score"] == "p_MSS_TMBH")
        & (results["beta_tissue_adjusted"] < 0)
        & (results["FDR_within_analysis"] < FDR_CUTOFF)
    ].copy()

    sensitivity = results[
        (results["analysis_set"] == "exclude_source_tissue")
        & (results["score"] == "p_MSS_TMBH")
    ][
        [
            "classifier_source",
            "drug_id",
            "beta_tissue_adjusted",
            "pvalue",
            "FDR_within_analysis",
            "n_models",
            "n_tissues",
        ]
    ].rename(columns={
        "beta_tissue_adjusted": "beta_excluding_source_tissue",
        "pvalue": "pvalue_excluding_source_tissue",
        "FDR_within_analysis": "FDR_excluding_source_tissue",
        "n_models": "n_models_excluding_source_tissue",
        "n_tissues": "n_tissues_excluding_source_tissue",
    })

    candidates = primary.merge(
        sensitivity,
        on=["classifier_source", "drug_id"],
        how="left",
    )
    candidates["same_direction_without_source_tissue"] = (
        candidates["beta_excluding_source_tissue"] < 0
    )
    return candidates.sort_values(
        [
            "same_direction_without_source_tissue",
            "FDR_within_analysis",
            "beta_tissue_adjusted",
        ],
        ascending=[False, True, True],
    )


specific_candidates = build_continuous_candidates(
    continuous_results
)
display(specific_candidates.head(50))

atomic_to_csv(
    specific_candidates,
    OUTPUT_DIR / "mss_tmbh_specific_candidate_drugs.csv",
    index=False,
)

In [ ]:
def flatten_classifier_columns(pivot):
    """Flatten MultiIndex columns created by pivot_table."""
    pivot.columns = [
        f"{metric}_{source}"
        for metric, source in pivot.columns
    ]
    return pivot.reset_index()


def add_cross_classifier_summary(pivot):
    """
    Add direction and significance agreement columns when both
    CRC and UCEC estimates are available.
    """
    crc_beta = "beta_tissue_adjusted_CRC"
    ucec_beta = "beta_tissue_adjusted_UCEC"

    crc_fdr = "FDR_within_analysis_CRC"
    ucec_fdr = "FDR_within_analysis_UCEC"

    if crc_beta in pivot.columns and ucec_beta in pivot.columns:
        both_available = (
            pivot[crc_beta].notna()
            & pivot[ucec_beta].notna()
        )

        pivot["same_direction"] = (
            both_available
            & (
                np.sign(pivot[crc_beta])
                == np.sign(pivot[ucec_beta])
            )
        )

        pivot["both_more_sensitive"] = (
            both_available
            & (pivot[crc_beta] < 0)
            & (pivot[ucec_beta] < 0)
        )

        pivot["both_more_resistant"] = (
            both_available
            & (pivot[crc_beta] > 0)
            & (pivot[ucec_beta] > 0)
        )

    if crc_fdr in pivot.columns and ucec_fdr in pivot.columns:
        pivot["significant_CRC"] = (
            pivot[crc_fdr] < 0.05
        )

        pivot["significant_UCEC"] = (
            pivot[ucec_fdr] < 0.05
        )

        pivot["significant_both"] = (
            pivot["significant_CRC"]
            & pivot["significant_UCEC"]
        )

        pivot["significant_either"] = (
            pivot["significant_CRC"]
            | pivot["significant_UCEC"]
        )

    sort_columns = [
        column
        for column in [
            "significant_both",
            "both_more_sensitive",
            "same_direction",
            "significant_either",
        ]
        if column in pivot.columns
    ]

    if sort_columns:
        pivot = pivot.sort_values(
            sort_columns,
            ascending=[False] * len(sort_columns),
        )

    return pivot.reset_index(drop=True)

def cross_classifier_hard_table(
    results,
    contrast,
    analysis_set="all_lineages",
):
    if results.empty:
        return pd.DataFrame()

    required_columns = {
        "analysis_set",
        "contrast",
        "classifier_source",
        "drug_id",
        "drug",
        "beta_tissue_adjusted",
        "FDR_within_analysis",
    }

    missing = required_columns - set(results.columns)

    if missing:
        raise KeyError(
            f"Hard-group results are missing columns: "
            f"{sorted(missing)}"
        )

    subset = results[
        (results["analysis_set"] == analysis_set)
        & (results["contrast"] == contrast)
    ].copy()

    if subset.empty:
        return pd.DataFrame()

    
    
    duplicate_keys = [
        "drug_id",
        "classifier_source",
        "contrast",
    ]

    duplicated = subset.duplicated(
        duplicate_keys,
        keep=False,
    )

    if duplicated.any():
        display(
            subset.loc[
                duplicated,
                duplicate_keys
                + [
                    "selection_rule",
                    "selection_threshold",
                    "selection_margin_threshold",
                    "pvalue",
                ],
            ].sort_values(duplicate_keys)
        )

        raise ValueError(
            "Multiple hard-group results exist for the same "
            "drug and classifier. Restrict the table to one "
            "selection run before pivoting."
        )

    value_columns = [
        "beta_tissue_adjusted",
        "FDR_within_analysis",
        "pvalue",
        "n_MSS_TMBH",
        "n_comparator",
        "n_tissues",
        "median_auc_difference",
    ]

    value_columns = [
        column
        for column in value_columns
        if column in subset.columns
    ]

    pivot = subset.pivot_table(
        index=["drug_id", "drug"],
        columns="classifier_source",
        values=value_columns,
        aggfunc="first",
    )

    pivot = flatten_classifier_columns(pivot)
    pivot = add_cross_classifier_summary(pivot)

    return pivot

cross_mss = cross_classifier_hard_table(
    primary_hard_results,
    contrast="MSS/TMB-H vs MSS/TMB-L",
)

cross_msi = cross_classifier_hard_table(
    primary_hard_results,
    contrast="MSS/TMB-H vs MSI-H",
)

display(cross_mss.head(40))
display(cross_msi.head(40))

atomic_to_csv(
    cross_mss,
    OUTPUT_DIR / "cross_cancer_vs_mss_tmbl.csv",
    index=False,
)

atomic_to_csv(
    cross_msi,
    OUTPUT_DIR / "cross_cancer_vs_msih.csv",
    index=False,
)

In [ ]:
def copy_source_or_write(filename, fallback):
    source = SOURCE_OUTPUT_DIR / filename
    destination = OUTPUT_DIR / filename

    if source.exists():
        shutil.copy2(source, destination)
        print("Copied:", filename)
    else:
        atomic_to_csv(fallback, destination, index=False)
        print("Created:", filename)


atomic_to_csv(
    classifier_performance,
    OUTPUT_DIR / "classifier_performance.csv",
    index=False,
)

copy_source_or_write(
    "crc_classifier_tuning.csv",
    crc_analysis["tuning"],
)
copy_source_or_write(
    "ucec_classifier_tuning.csv",
    ucec_analysis["tuning"],
)
copy_source_or_write(
    "crc_tumor_oof_predictions.csv",
    crc_analysis["oof"],
)
copy_source_or_write(
    "ucec_tumor_oof_predictions.csv",
    ucec_analysis["oof"],
)

run_manifest = pd.DataFrame([{
    "source_output_directory": str(SOURCE_OUTPUT_DIR),
    "pan_cancer_output_directory": str(OUTPUT_DIR),
    "celligner_aligned_path": str(CELLIGNER_ALIGNED_PATH),
    "classifier_cache_directory": str(CACHE_DIR),
    "tissue_column": TISSUE_COLUMN,
    "n_unique_aligned_models": all_cell_predictions["DepMap_ID"].nunique(),
    "n_unique_PRISM_models": prism_collapsed["depmap_id"].nunique(),
    "primary_probability_threshold": PRIMARY_PROBABILITY_THRESHOLD,
    "primary_margin_threshold": PRIMARY_MARGIN_THRESHOLD,
    "min_models_per_tissue": MIN_MODELS_PER_TISSUE,
    "min_total_models_continuous": MIN_TOTAL_MODELS_CONTINUOUS,
    "min_tissues_continuous": MIN_TISSUES_CONTINUOUS,
}])

atomic_to_csv(
    run_manifest,
    OUTPUT_DIR / "pan_cancer_run_manifest.csv",
    index=False,
)

print("\nOutputs written to:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path.name)